# 05 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific champions from `data/champions.json` (generated by Notebook 04).
2. Executes each champion **N=10 independent times** on its target problem only (`f1, f8, f11, f15, f21`).
3. Uses the **exact same noisy `BBOBProblem` wrapper** (`MultiplicativeNoiseStrategy(0.05)`) as baselines.
4. Attaches IOH Analyzer via `problem.attach_analyzer(...)` to output IOH `.dat` performance files to `data/ioh_logs/f{p_id}_5D_std0.05/llamea_champion/`.

In [ ]:
import sys
import json
import numpy as np
from pathlib import Path

# Add src to path
PROJECT_ROOT = Path('../').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from infra.problems.bbob import BBOBProblem
from domain.services.noise_strategy import MultiplicativeNoiseStrategy
from synthesis.executor import AlgorithmExecutor

CHAMPIONS_PATH = PROJECT_ROOT / 'data' / 'champions.json'
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'
N_RUNS = 10
BUDGET = 100000
NOISE_STD = 0.05
DIM = 5
TIMEOUT_SECONDS = 30.0

print(f'Champions JSON: {CHAMPIONS_PATH}')
print(f'IOH Logs Output: {IOH_LOGS_DIR}')
print(f'Runs per champion: {N_RUNS}')
print(f'Budget: {BUDGET} evaluations')

## 1. Load Champions JSON

In [ ]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 04 first.')

with open(CHAMPIONS_PATH, 'r') as f:
    champions = json.load(f)

print(f'Loaded {len(champions)} champion configuration(s):')
for pid, info in champions.items():
    print(f"  f{pid}: {info['algorithm_name']} (from Exp #{info['experiment_id']})")

## 2. Execute Champion Evaluation Benchmark

In [ ]:
executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

for pid_str, info in champions.items():
    p_id = int(pid_str)
    code_file = PROJECT_ROOT / info['code_path'] if not Path(info['code_path']).is_absolute() else Path(info['code_path'])
    
    if not code_file.exists():
        print(f'[WARN] Code file for f{p_id} not found at {code_file}. Skipping.')
        continue
        
    code_content = code_file.read_text(encoding='utf-8')
    algo_name = info['algorithm_name']
    out_dir = IOH_LOGS_DIR / f'f{p_id}_{DIM}D_std{NOISE_STD}'
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f'\n=== Evaluating Champion for f{p_id}: {algo_name} (N={N_RUNS} runs, Budget={BUDGET}) ===')
    
    # Instantiate problem wrapper
    problem = BBOBProblem(
        problem_id=p_id,
        dim=DIM,
        instance_id=1,
        noise_strategy=MultiplicativeNoiseStrategy(NOISE_STD),
    )
    
    # Attach IOH logger directly to problem interface
    problem.attach_analyzer(
        log_dir=out_dir,
        folder_name='llamea_champion',
        algorithm_name='LLaMEA'
    )
    
    clean_errors = []
    for run_idx in range(1, N_RUNS + 1):
        problem.reset()
        try:
            best_x, best_y = executor.execute_algorithm(
                code=code_content,
                name=algo_name,
                dim=DIM,
                problem=problem.get_objective_fn(),
                budget=BUDGET,
            )
            
            if best_x is not None:
                clean_val = problem.eval_clean(best_x)
                clean_err = abs(clean_val - problem.true_optimum)
                clean_errors.append(clean_err)
                evals = problem.evaluations
                print(f'  Run {run_idx:2d}/{N_RUNS}: evals={evals}, final clean error={clean_err:.6e}')
            else:
                print(f'  Run {run_idx:2d}/{N_RUNS}: returned best_x is None')
        except Exception as e:
            print(f'  Run {run_idx:2d}/{N_RUNS} execution failed: {e}')
            
    # Safely close logger
    problem.close_logger()
    
    if clean_errors:
        med_err = float(np.median(clean_errors))
        print(f'  f{p_id} Champion Median Clean Error across {len(clean_errors)} runs: {med_err:.6e}')
        
print('\nChampion evaluations complete!')